In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/raw/2017-2020")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
files = {
    "demo": "P_DEMO.xpt",
    "body": "P_BMX.xpt",
    "bp": "P_BPXO.xpt",
    "liver": "P_LUX.xpt",
    "hdl": "P_HDL.xpt",
    "triglycerides": "P_TRIGLY.xpt",
    "glucose": "P_GLU.xpt",
    "diabetes": "P_DIQ.xpt",
}

data = {
    name: pd.read_sas(DATA_DIR / filename, format="xport")
    for name, filename in files.items()
}

In [4]:
for name, df in data.items():
    print(f"{name:15} {df.shape}")

demo            (15560, 29)
body            (14300, 22)
bp              (11656, 12)
liver           (10409, 13)
hdl             (12198, 3)
triglycerides   (5090, 10)
glucose         (5090, 4)
diabetes        (14986, 28)


In [11]:
# Rename NHANES variables to human-readable names

rename_map = {
    # Demographics
    "RIDAGEYR": "age",                  # Age in years
    "RIAGENDR": "sex",                  # Sex
    "RIDRETH1": "race_ethnicity",       # Race/ethnicity category

    # Body measurements
    "BMXBMI": "bmi",                    # Body mass index (kg/m^2)
    "BMXWAIST": "waist_cm",             # Waist circumference (cm)

    # Blood pressure
    "BPXOSY1": "systolic_bp_1",         # Systolic BP, reading 1 (mmHg)
    "BPXOSY2": "systolic_bp_2",         # Systolic BP, reading 2 (mmHg)
    "BPXOSY3": "systolic_bp_3",         # Systolic BP, reading 3 (mmHg)
    "BPXODI1": "diastolic_bp_1",        # Diastolic BP, reading 1 (mmHg)
    "BPXODI2": "diastolic_bp_2",        # Diastolic BP, reading 2 (mmHg)
    "BPXODI3": "diastolic_bp_3",        # Diastolic BP, reading 3 (mmHg)

    # Liver elastography
    "LUXCAPM": "cap_db_m",              # Median CAP; liver fat measure (dB/m)
    "LUXCPIQR": "cap_iqr",              # CAP interquartile range

    # Laboratory measurements
    "LBDHDD": "hdl_mg_dl",              # HDL cholesterol (mg/dL)
    "LBXTR": "triglycerides_mg_dl",     # Triglycerides (mg/dL)
    "LBXGLU": "glucose_mg_dl",          # Fasting glucose (mg/dL)

    # Diabetes
    "DIQ010": "diabetes_status",         # Doctor-diagnosed diabetes response

    # Survey design
    "WTSAFPRP": "fasting_weight",        # Fasting subsample survey weight
}

for name in data:
    data[name] = data[name].rename(columns=rename_map)

In [13]:
demo = data["demo"][
    ["SEQN", "age", "sex", "race_ethnicity"]
].copy()

body = data["body"][
    ["SEQN", "bmi", "waist_cm"]
].copy()

liver = data["liver"][
    ["SEQN", "cap_db_m", "cap_iqr"]
].copy()

hdl = data["hdl"][
    ["SEQN", "hdl_mg_dl"]
].copy()

triglycerides = data["triglycerides"][
    ["SEQN", "triglycerides_mg_dl", "fasting_weight"]
].copy()

glucose = data["glucose"][
    ["SEQN", "glucose_mg_dl"]
].copy()

diabetes = data["diabetes"][
    ["SEQN", "diabetes_status"]
].copy()

bp = data["bp"][
    [
        "SEQN",
        "systolic_bp_1", "systolic_bp_2", "systolic_bp_3",
        "diastolic_bp_1", "diastolic_bp_2", "diastolic_bp_3",
    ]
].copy()

bp["systolic_bp"] = bp[
    ["systolic_bp_1", "systolic_bp_2", "systolic_bp_3"]
].mean(axis=1)

bp["diastolic_bp"] = bp[
    ["diastolic_bp_1", "diastolic_bp_2", "diastolic_bp_3"]
].mean(axis=1)

bp = bp[["SEQN", "systolic_bp", "diastolic_bp"]]